In [1]:
#Use Python 3.9.12
#pip install ipyparallel
#pip install markovify

In [2]:
import pandas as pd
import numpy as np
import os 
from os import walk
import re
import time
import ipyparallel as ipp

In [3]:
#Load your AD Sets here
AD_Seq_Complete_List = pd.read_csv('Melvin_AD_Seq_Complete_List(V3).csv')
AD_Seq_Complete_List = AD_Seq_Complete_List[AD_Seq_Complete_List['AD Origin'] == 'More_Plant_data.PADI_library.xlsx']

In [4]:
AD_Seq_Complete_List

,AD Sequence,AD Seq Length,AD Activity Score,AD Name,AD Origin
8704,MEDQVGFGFRPNDEELVGHYLRNKIEGNTSRDVEVAISEV,40,1.004699,AT1G01010.1_1,More_Plant_data.PADI_library.xlsx
8705,QFNENSNIMQQQPLQGSFNPLLEYDFANHGGQWLSDYIDL,40,6.891245,AT1G01010.1_211,More_Plant_data.PADI_library.xlsx
8706,QQPLQGSFNPLLEYDFANHGGQWLSDYIDLQQQVPYLAPY,40,5.023688,AT1G01010.1_221,More_Plant_data.PADI_library.xlsx
8707,LLEYDFANHGGQWLSDYIDLQQQVPYLAPYENESEMIWKH,40,6.381243,AT1G01010.1_231,More_Plant_data.PADI_library.xlsx
8708,GQWLSDYIDLQQQVPYLAPYENESEMIWKHVIEENFEFLV,40,5.071580,AT1G01010.1_241,More_Plant_data.PADI_library.xlsx
...,...,...,...,...,...
14906,RNGHNLGLSLGESSSEEFRLKSDIASIRSRIRDEERLLGS,40,1.682533,AT5G67180.1_261,More_Plant_data.PADI_library.xlsx
14907,NFISRLRSLYNYMWIPYDTAEMFLTRGSEQRQWYEADISW,40,1.859266,AT5G67411.1_141,More_Plant_data.PADI_library.xlsx
14908,NYMWIPYDTAEMFLTRGSEQRQWYEADISWKIDNVVAKEG,40,2.150537,AT5G67411.1_151,More_Plant_data.PADI_library.xlsx
14909,TPSSTRRRSVTPSMNSEDSGTTTTTTAFCDKGDVYGNGGG,40,1.961084,AT5G67420.1_201,More_Plant_data.PADI_library.xlsx


In [5]:
#Format your list of ADs into a string that can be fed into Markovify
ADs_In_A_String = ''
for i in AD_Seq_Complete_List['AD Sequence']:
    ADs_In_A_String = ADs_In_A_String + (" ".join(i)) + "\n"

In [6]:
#Creating Markovify models with varying state sizes for varying libraries
import markovify

SS1 = markovify.NewlineText(ADs_In_A_String,state_size=1)
SS2 = markovify.NewlineText(ADs_In_A_String,state_size=2)
SS3 = markovify.NewlineText(ADs_In_A_String,state_size=3)
SS4 = markovify.NewlineText(ADs_In_A_String,state_size=4)
SS5 = markovify.NewlineText(ADs_In_A_String,state_size=5)
SS6 = markovify.NewlineText(ADs_In_A_String,state_size=6)
SS7 = markovify.NewlineText(ADs_In_A_String,state_size=7)
SS8 = markovify.NewlineText(ADs_In_A_String,state_size=8)

In [7]:
ipp.__version__   # check that at least version 7
mycluster = ipp.Cluster(n = int(os.getenv('SLURM_CPUS_ON_NODE')))
mycluster.start_cluster_sync()

c = mycluster.connect_client_sync()
c.wait_for_engines(n = int(os.getenv('SLURM_CPUS_ON_NODE')))
c.ids

Starting 28 engines with <class 'ipyparallel.cluster.launcher.LocalEngineSetLauncher'>


  0%|          | 0/28 [00:00<?, ?engine/s]

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27]

In [8]:
dview = c[:]
dview.execute('import markovify')
# we need to broadcast those data objects to the workers

mydict = dict(
classSS1 = SS1,
classSS2 = SS2,
classSS3 = SS3,
classSS4 = SS4,
classSS5 = SS5,
classSS6 = SS6,
classSS7 = SS7,
classSS8 = SS8)
dview.push(mydict)

<AsyncResult(_push): pending>

In [9]:
def remove_spaces(string):
    newstring = []
    for i in range(len(string)):
        newstring.append(string[i].replace(" ", ""))
    return newstring

In [10]:
%%time
# Need a wrapper function because map() only operates on one argument
def wrapper(i):
    return classSS5.make_short_sentence(max_overlap_ratio=70, min_chars=79, max_chars=80, tries=50000)

n = 50000

# Run a parallel map, executing the wrapper function on indices 0,...,n-1
lview = c.load_balanced_view()

# Cause execution on main process to wait while tasks sent to workers finish
lview.block = True 
out = lview.map(wrapper, range(n))   # Run calculation in parallel
out = remove_spaces(out)

tempdf = pd.DataFrame(out)
tempdf.to_csv('SS5_50_overlap.csv', index = False)


CPU times: user 2min 6s, sys: 28.6 s, total: 2min 35s
Wall time: 1h 13min 32s


In [11]:
%%time
# Need a wrapper function because map() only operates on one argument
def wrapper(i):
    return classSS5.make_short_sentence(max_overlap_ratio=70, min_chars=79, max_chars=80, tries=50000)

n = 50000

# Run a parallel map, executing the wrapper function on indices 0,...,n-1
lview = c.load_balanced_view()

# Cause execution on main process to wait while tasks sent to workers finish
lview.block = True 
out = lview.map(wrapper, range(n))   # Run calculation in parallel
out = remove_spaces(out)

tempdf = pd.DataFrame(out)
tempdf.to_csv('SS5_70_overlap.csv', index = False)


CPU times: user 2min 10s, sys: 27.5 s, total: 2min 38s
Wall time: 1h 13min 22s


In [ ]:
%%time
# Need a wrapper function because map() only operates on one argument
def wrapper(i):
    return classSS6.make_short_sentence(max_overlap_ratio=10, min_chars=79, max_chars=80, tries=50000)

n = 50000

# Run a parallel map, executing the wrapper function on indices 0,...,n-1
lview = c.load_balanced_view()

# Cause execution on main process to wait while tasks sent to workers finish
lview.block = True 
out = lview.map(wrapper, range(n))   # Run calculation in parallel
out = remove_spaces(out)

tempdf = pd.DataFrame(out)
tempdf.to_csv('SS6_10_overlap.csv', index = False)


In [ ]:
%%time
# Need a wrapper function because map() only operates on one argument
def wrapper(i):
    return classSS6.make_short_sentence(max_overlap_ratio=20, min_chars=79, max_chars=80, tries=50000)

n = 50000

# Run a parallel map, executing the wrapper function on indices 0,...,n-1
lview = c.load_balanced_view()

# Cause execution on main process to wait while tasks sent to workers finish
lview.block = True 
out = lview.map(wrapper, range(n))   # Run calculation in parallel
out = remove_spaces(out)

tempdf = pd.DataFrame(out)
tempdf.to_csv('SS6_20_overlap.csv', index = False)


In [ ]:
%%time
# Need a wrapper function because map() only operates on one argument
def wrapper(i):
    return classSS6.make_short_sentence(max_overlap_ratio=50, min_chars=79, max_chars=80, tries=50000)

n = 50000

# Run a parallel map, executing the wrapper function on indices 0,...,n-1
lview = c.load_balanced_view()

# Cause execution on main process to wait while tasks sent to workers finish
lview.block = True 
out = lview.map(wrapper, range(n))   # Run calculation in parallel
out = remove_spaces(out)

tempdf = pd.DataFrame(out)
tempdf.to_csv('SS6_50_overlap.csv', index = False)


In [ ]:
%%time
# Need a wrapper function because map() only operates on one argument
def wrapper(i):
    return classSS6.make_short_sentence(max_overlap_ratio=70, min_chars=79, max_chars=80, tries=50000)

n = 50000

# Run a parallel map, executing the wrapper function on indices 0,...,n-1
lview = c.load_balanced_view()

# Cause execution on main process to wait while tasks sent to workers finish
lview.block = True 
out = lview.map(wrapper, range(n))   # Run calculation in parallel
out = remove_spaces(out)

tempdf = pd.DataFrame(out)
tempdf.to_csv('SS6_70_overlap.csv', index = False)


In [ ]:
%%time
# Need a wrapper function because map() only operates on one argument
def wrapper(i):
    return classSS3.make_short_sentence(max_overlap_ratio=50, min_chars=79, max_chars=80, tries=50000)

n = 50000

# Run a parallel map, executing the wrapper function on indices 0,...,n-1
lview = c.load_balanced_view()

# Cause execution on main process to wait while tasks sent to workers finish
lview.block = True 
out = lview.map(wrapper, range(n))   # Run calculation in parallel
out = remove_spaces(out)

tempdf = pd.DataFrame(out)
tempdf.to_csv('SS3_50_overlap.csv', index = False)
